# Fashion-MNIST 실습: 학습 곡선 · 하이퍼파라미터 실험 · DNN vs CNN

| 단계 | 내용 |
|---|---|
| 0 | 환경 설정 · 데이터 로드 · 공통 함수 |
| 1 | 베이스라인 DNN / CNN 학습 + **학습 곡선** |
| 2 | 실험 A: **Batch size** 변경 (32 / 128 / 512) |
| 3 | 실험 B: **학습률** 변경 (0.01 / 0.001 / 0.0001) |
| 4 | 실험 C: **DNN 파라미터를 늘리면 CNN을 따라잡을 수 있을까?** |
| 5 | 전체 결과 요약 · 결론 |

> **실행 전 준비**: Colab 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 를 선택하세요.
> 위에서부터 순서대로 실행하면 되고, GPU 기준 전체 실행에 약 **10분** 정도 걸립니다.
>
> 그래프의 제목·축은 Colab에서 한글 폰트가 깨지지 않도록 영어로 표시했습니다.

## 0. 환경 설정 · 데이터 로드

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42       # 모든 실험에서 같은 시드를 써야 공정한 비교가 됨
EPOCHS = 10     # 기본 에폭 수 (원본 노트북과 동일)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# 데이터 로드 + 정규화
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = (x_train / 255.0).astype('float32')
x_test  = (x_test  / 255.0).astype('float32')

# CNN용: 채널 차원 추가 (28, 28) -> (28, 28, 1)
x_train_cnn = np.expand_dims(x_train, -1)
x_test_cnn  = np.expand_dims(x_test, -1)

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print("train:", x_train.shape, " test:", x_test.shape)

### 공통 함수
- `build_dnn(hidden_units)`: 은닉층 크기를 바꿔 가며 DNN을 만들 수 있게 함수로 만들었습니다.
- `build_cnn()`: 원본 노트북의 CNN과 같은 구조입니다. (경고가 뜨지 않도록 `layers.Input` 사용)
- `run_experiment()`: 시드 고정 → 모델 생성 → 학습 → 평가 → 결과 기록까지 한 번에 처리합니다.

In [ ]:
def build_dnn(hidden_units=(256, 128), dropout=0.3):
    model = models.Sequential([layers.Input(shape=(28, 28)), layers.Flatten()])
    for units in hidden_units:
        model.add(layers.Dense(units, activation='relu'))
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(10, activation='softmax'))
    return model


def build_cnn():
    return models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])


class EpochTimer(tf.keras.callbacks.Callback):
    # 에폭마다 걸린 시간을 기록하는 콜백
    def on_train_begin(self, logs=None):
        self.times = []
    def on_epoch_begin(self, epoch, logs=None):
        self._t = time.time()
    def on_epoch_end(self, epoch, logs=None):
        self.times.append(time.time() - self._t)


all_results = []   # 모든 실험 결과를 모아 두는 리스트


def run_experiment(name, build_fn, x_tr, x_te, experiment,
                   batch_size=128, lr=1e-3, epochs=EPOCHS, callbacks=None):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    model = build_fn()
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    timer = EpochTimer()
    history = model.fit(x_tr, y_train, epochs=epochs, batch_size=batch_size,
                        validation_split=0.1, verbose=0,
                        callbacks=[timer] + (callbacks or []))
    test_loss, test_acc = model.evaluate(x_te, y_test, verbose=0)

    h = history.history
    # 첫 에폭은 GPU 초기화/그래프 컴파일 때문에 느리므로 평균에서 제외
    sec = np.mean(timer.times[1:]) if len(timer.times) > 1 else timer.times[0]
    result = {
        'experiment': experiment,
        'model': name,
        'params': model.count_params(),
        'batch_size': batch_size,
        'lr': lr,
        'epochs_run': len(h['accuracy']),
        'steps/epoch': int(np.ceil(len(x_tr) * 0.9 / batch_size)),
        'train_acc': h['accuracy'][-1],
        'val_acc': h['val_accuracy'][-1],
        'best_val_acc': max(h['val_accuracy']),
        'test_acc': test_acc,
        'test_loss': test_loss,
        'sec/epoch': sec,
    }
    all_results.append(result)
    print(f"[{name:<22}] params={result['params']:>10,}  "
          f"train={result['train_acc']:.4f}  val={result['val_acc']:.4f}  "
          f"test={test_acc:.4f}  ({sec:.1f}s/epoch)")
    return history, result


def plot_curves(histories, title, show_train=True):
    # 실선 = validation, 점선 = train
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
    for ax, metric in zip(axes, ['accuracy', 'loss']):
        for label, hist in histories.items():
            ep = range(1, len(hist.history[metric]) + 1)
            line, = ax.plot(ep, hist.history['val_' + metric], lw=2, marker='o',
                            ms=4, label=f'{label} (val)')
            if show_train:
                ax.plot(ep, hist.history[metric], '--', lw=1.5, alpha=0.6,
                        color=line.get_color(), label=f'{label} (train)')
        ax.set_title(f'{title} - {metric}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


def show_table(results):
    cols = ['model', 'params', 'batch_size', 'lr', 'steps/epoch',
            'train_acc', 'val_acc', 'test_acc', 'sec/epoch']
    df = pd.DataFrame(results)[cols]
    return df.style.format({'params': '{:,}', 'lr': '{:g}', 'train_acc': '{:.4f}',
                            'val_acc': '{:.4f}', 'test_acc': '{:.4f}',
                            'sec/epoch': '{:.2f}'}).hide(axis='index')

## 1. 베이스라인 학습 + 학습 곡선

원본 노트북과 같은 조건(batch 128, Adam 학습률 0.001, 10 에폭)으로 DNN과 CNN을 학습합니다.

In [ ]:
h_dnn, r_dnn = run_experiment('DNN (256-128)', build_dnn, x_train, x_test, experiment='baseline')
h_cnn, r_cnn = run_experiment('CNN', build_cnn, x_train_cnn, x_test_cnn, experiment='baseline')

plot_curves({'DNN': h_dnn, 'CNN': h_cnn}, 'Baseline')

### 📈 학습 곡선 읽는 법
- **실선 = validation, 점선 = train** 입니다.
- **두 선이 붙어서 같이 좋아지면** 잘 학습되고 있는 것입니다.
- **train은 계속 좋아지는데 val이 멈추거나 나빠지면** 과적합이 시작된 것입니다. (loss 그래프에서 val 곡선이 다시 올라가는 지점)
- 초반에 **train 정확도가 val보다 낮은** 것은 Dropout 때문입니다. 학습할 때만 뉴런을 꺼서 일부러 불리하게 학습하고, 검증할 때는 Dropout이 꺼지기 때문이에요.
- CNN의 val 곡선이 DNN보다 **항상 위에 있는지**, 10 에폭 시점에서 **아직 오르는 중인지**(에폭을 늘리면 더 좋아질 여지) 확인해 보세요.

## 2. 실험 A: Batch size 변경

CNN으로 **batch size만** 32 / 128 / 512로 바꾸고 나머지(학습률 0.001, 10 에폭)는 고정합니다.

**확인할 점**
- batch가 작으면 한 에폭에 가중치를 더 **자주** 갱신합니다. (`steps/epoch` 열 참고: 32 → 1,688번, 512 → 106번)
- 대신 한 번에 처리하는 양이 적어서 GPU를 덜 활용하므로 **에폭당 시간**이 길어집니다.
- 같은 10 에폭이라면 batch가 너무 크면 갱신 횟수가 부족해서 덜 학습될 수 있습니다.

In [ ]:
batch_sizes = [32, 128, 512]
hist_bs, res_bs = {}, []

for bs in batch_sizes:
    h, r = run_experiment(f'CNN bs={bs}', build_cnn, x_train_cnn, x_test_cnn,
                          experiment='A: batch size', batch_size=bs)
    hist_bs[f'bs={bs}'] = h
    res_bs.append(r)

plot_curves(hist_bs, 'Batch size')
show_table(res_bs)

In [ ]:
# 정확도와 속도를 나란히 비교
labels = [f'bs={bs}' for bs in batch_sizes]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

acc = [r['test_acc'] for r in res_bs]
axes[0].bar(labels, acc, color='#4C72B0')
axes[0].set_ylim(min(acc) - 0.02, max(acc) + 0.01)
axes[0].set_title('Test accuracy by batch size')
for i, v in enumerate(acc):
    axes[0].text(i, v + 0.001, f'{v:.4f}', ha='center')

sec = [r['sec/epoch'] for r in res_bs]
axes[1].bar(labels, sec, color='#DD8452')
axes[1].set_title('Seconds per epoch by batch size')
for i, v in enumerate(sec):
    axes[1].text(i, v, f'{v:.1f}s', ha='center', va='bottom')

plt.tight_layout()
plt.show()

### ✏️ 실험 A 정리 (실행 결과를 보고 작성)
- 가장 정확도가 높았던 batch size: ______
- 가장 빨랐던 batch size: ______
- batch가 작을 때 학습 곡선이 더 **들쭉날쭉**했는지? (작은 batch는 기울기 추정에 노이즈가 많음)
- 결론: 속도와 정확도 사이에 어떤 트레이드오프가 있었는지 한두 문장으로 정리해 보세요.

## 3. 실험 B: 학습률(Learning rate) 변경

CNN으로 **학습률만** 0.01 / 0.001 / 0.0001로 바꾸고 batch size는 128로 고정합니다.

**확인할 점**
- **너무 크면(0.01)**: 최솟값 근처에서 크게 튀어서 loss가 불안정하거나, 오히려 성능이 떨어질 수 있습니다.
- **적당하면(0.001)**: Adam의 기본값으로, 대부분 안정적으로 수렴합니다.
- **너무 작으면(0.0001)**: 안정적이지만 느리게 학습해서 10 에폭 안에 다 수렴하지 못할 수 있습니다.

In [ ]:
learning_rates = [1e-2, 1e-3, 1e-4]
hist_lr, res_lr = {}, []

for lr in learning_rates:
    h, r = run_experiment(f'CNN lr={lr:g}', build_cnn, x_train_cnn, x_test_cnn,
                          experiment='B: learning rate', lr=lr)
    hist_lr[f'lr={lr:g}'] = h
    res_lr.append(r)

plot_curves(hist_lr, 'Learning rate')
show_table(res_lr)

### ✏️ 실험 B 정리 (실행 결과를 보고 작성)
- lr=0.01의 loss 곡선이 불안정했는지, 정확도가 기본값보다 낮았는지: ______
- lr=0.0001은 10 에폭 끝에서 **아직 곡선이 오르는 중**이었는지: ______ (그렇다면 에폭을 늘리면 따라잡을 가능성이 있음)
- 결론: 이 문제에서 적절한 학습률은 ______ 이고, 이유는 ______.

## 4. 실험 C: DNN 파라미터를 늘리면 CNN을 따라잡을 수 있을까?

은닉층의 크기와 개수를 늘려 DNN의 파라미터 수를 약 10만 개에서 약 420만 개까지 키워 봅니다.
CNN(약 24만 개)과 같은 조건(batch 128, lr 0.001, 10 에폭, Dropout 0.3)에서 비교합니다.

| 모델 | 은닉층 구성 |
|---|---|
| DNN-S | 128 |
| DNN-M | 256-128 (베이스라인) |
| DNN-L | 512-256 |
| DNN-XL | 1024-512-256 |
| DNN-XXL | 2048-1024-512 |

In [ ]:
dnn_configs = {
    'DNN-S (128)':             (128,),
    'DNN-M (256-128)':         (256, 128),
    'DNN-L (512-256)':         (512, 256),
    'DNN-XL (1024-512-256)':   (1024, 512, 256),
    'DNN-XXL (2048-1024-512)': (2048, 1024, 512),
}
hist_dnn, res_dnn = {}, []

for name, units in dnn_configs.items():
    h, r = run_experiment(name, lambda u=units: build_dnn(u), x_train, x_test,
                          experiment='C: DNN scaling')
    hist_dnn[name] = h
    res_dnn.append(r)

show_table(res_dnn + [r_cnn])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) 파라미터 수 vs Test 정확도
params = [r['params'] for r in res_dnn]
accs = [r['test_acc'] for r in res_dnn]
axes[0].plot(params, accs, 'o-', lw=2, ms=8, label='DNN')
for r in res_dnn:
    axes[0].annotate(r['model'].split(' ')[0], (r['params'], r['test_acc']),
                     textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
axes[0].scatter(r_cnn['params'], r_cnn['test_acc'], marker='*', s=350,
                color='crimson', zorder=5, label='CNN')
axes[0].axhline(r_cnn['test_acc'], color='crimson', ls='--', alpha=0.5)
axes[0].set_xscale('log')
axes[0].set_xlabel('Number of parameters (log scale)')
axes[0].set_ylabel('Test accuracy')
axes[0].set_title('Params vs Test accuracy')
axes[0].grid(alpha=0.3)
axes[0].legend()

# (2) Train vs Test 정확도 (과적합 정도)
names = [r['model'].split(' ')[0] for r in res_dnn] + ['CNN']
train = [r['train_acc'] for r in res_dnn] + [r_cnn['train_acc']]
test = [r['test_acc'] for r in res_dnn] + [r_cnn['test_acc']]
x = np.arange(len(names))
axes[1].bar(x - 0.2, train, 0.4, label='train')
axes[1].bar(x + 0.2, test, 0.4, label='test')
axes[1].set_xticks(x)
axes[1].set_xticklabels(names)
axes[1].set_ylim(0.8, 1.0)
axes[1].set_title('Train vs Test accuracy')
axes[1].grid(alpha=0.3, axis='y')
axes[1].legend()

plt.tight_layout()
plt.show()

best_dnn = max(res_dnn, key=lambda r: r['test_acc'])
print(f"가장 좋은 DNN : {best_dnn['model']}  test={best_dnn['test_acc']:.4f}  params={best_dnn['params']:,}")
print(f"CNN          : test={r_cnn['test_acc']:.4f}  params={r_cnn['params']:,}")
print(f"차이          : {r_cnn['test_acc'] - best_dnn['test_acc']:+.4f}  "
      f"(파라미터는 DNN이 {best_dnn['params'] / r_cnn['params']:.1f}배)")

### 보너스: 에폭을 넉넉히 주면 따라잡을까?

큰 DNN은 10 에폭이 부족했을 수도 있으니, 가장 큰 DNN에 **최대 30 에폭 + EarlyStopping**을 줘서 한 번 더 비교합니다.
EarlyStopping은 val_loss가 5 에폭 동안 좋아지지 않으면 학습을 멈추고, 가장 좋았던 시점의 가중치로 되돌립니다.

In [ ]:
es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                      restore_best_weights=True)
h_xxl30, r_xxl30 = run_experiment('DNN-XXL (30ep+ES)',
                                  lambda: build_dnn((2048, 1024, 512)),
                                  x_train, x_test, experiment='C: DNN scaling (bonus)',
                                  epochs=30, callbacks=[es])
print(f"실제 학습한 에폭: {r_xxl30['epochs_run']}")
print(f"CNN과의 차이: {r_cnn['test_acc'] - r_xxl30['test_acc']:+.4f}")

plot_curves({'DNN-XXL (30ep+ES)': h_xxl30, 'CNN (10ep)': h_cnn}, 'Bigger DNN with more epochs')

### ✏️ 실험 C 정리 (실행 결과를 보고 작성)
- 파라미터를 늘릴수록 DNN 정확도가 계속 올랐는지, 어느 지점부터 **거의 오르지 않았는지(포화)**: ______
- 가장 큰 DNN(파라미터 약 17배)도 CNN을 넘었는지: ______
- 큰 DNN일수록 train과 test의 차이(과적합)가 커졌는지: ______
- 결론을 쓸 때 참고할 포인트:
  - DNN은 Flatten으로 이미지를 펼치는 순간 **픽셀 사이의 공간 관계를 잃어버립니다.** 파라미터를 늘려도 이 정보를 처음부터 다시 배워야 해서 효율이 떨어집니다.
  - CNN은 **합성곱(지역 패턴) + 가중치 공유 + 풀링(위치 불변성)** 이라는 이미지에 맞는 구조를 처음부터 갖고 있습니다. 그래서 적은 파라미터로도 더 잘합니다.
  - 즉 **모델 크기보다 데이터에 맞는 구조(inductive bias)가 더 중요하다**는 것을 보여 주는 실험입니다.

## 5. 전체 결과 요약

In [ ]:
summary = pd.DataFrame(all_results)
cols = ['experiment', 'model', 'params', 'batch_size', 'lr', 'epochs_run',
        'train_acc', 'val_acc', 'test_acc', 'sec/epoch']
display(summary[cols].style.format({'params': '{:,}', 'lr': '{:g}', 'train_acc': '{:.4f}',
                                     'val_acc': '{:.4f}', 'test_acc': '{:.4f}',
                                     'sec/epoch': '{:.2f}'}).hide(axis='index'))

summary.to_csv('fashion_mnist_results.csv', index=False)
print("결과를 fashion_mnist_results.csv로 저장했습니다. (왼쪽 파일 탭에서 다운로드 가능)")

### ✏️ 최종 결론 (실행 결과를 보고 작성)
1. **학습 곡선**: DNN과 CNN 모두 과적합은 ______ 했고, CNN은 10 에폭 시점에 ______.
2. **Batch size**: ______ 가 정확도/속도 면에서 가장 균형이 좋았다.
3. **학습률**: ______ 가 가장 적절했고, 너무 크거나 작을 때는 ______ 현상이 나타났다.
4. **DNN vs CNN**: DNN 파라미터를 약 ___배까지 늘려도 CNN의 정확도(______)를 따라잡지 못했다 / 따라잡았다. 그 이유는 ______.

> 참고: 시드를 고정했지만 GPU 연산 특성상 실행할 때마다 소수점 셋째 자리 정도는 달라질 수 있습니다.